In [3]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_FILENAME = "dataset - 2020-09-24.csv"
DATA_PATH = Path.cwd() / DATA_FILENAME
if not DATA_PATH.exists():
    DATA_PATH = Path.cwd().parent / "dataset" / DATA_FILENAME

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"ไม่พบไฟล์ {DATA_FILENAME!r}; ตรวจสอบว่าไฟล์อยู่ในโฟลเดอร์ dataset"
    )

raw_df = pd.read_csv(DATA_PATH)
df = raw_df.copy()

# Normalize text fields and convert blank strings to missing values.
text_columns = df.columns[df.dtypes == "object"]
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()
df = df.replace({"": pd.NA, "nan": pd.NA})

# Percentage columns are stored as strings such as "78%" in the source file.
percentage_columns = [
    "Tackle success %", "Shooting accuracy %", "Cross accuracy %"
]
for column in percentage_columns:
    df[column] = (
        df[column].astype("string").str.rstrip("%").replace("<NA>", pd.NA)
    )
    df[column] = pd.to_numeric(df[column], errors="coerce") / 100

# Convert every remaining measurable field to numeric values.
non_numeric_columns = {"Name", "Club", "Position", "Nationality"}
for column in df.columns:
    if column not in non_numeric_columns and column not in percentage_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

# These fields represent counts and are blank for positions where they do not apply.
count_columns = [
    "Headed goals", "Goals with right foot", "Goals with left foot",
    "Penalties scored", "Freekicks scored", "Shots", "Shots on target",
    "Hit woodwork", "Big chances missed", "Tackles", "Last man tackles",
    "Blocked shots", "Interceptions", "Clearances", "Headed Clearance",
    "Clearances off line", "Recoveries", "Duels won", "Duels lost",
    "Successful 50/50s", "Aerial battles won", "Aerial battles lost",
    "Own goals", "Errors leading to goal", "Assists", "Big chances created",
    "Crosses", "Through balls", "Accurate long balls", "Yellow cards",
    "Red cards", "Fouls", "Offsides"
]
for column in count_columns:
    if column in df:
        df[column] = df[column].fillna(0)

print(f"Loaded {len(df):,} players and {len(df.columns)} columns")
print(f"Duplicate rows: {df.duplicated().sum():,}")
print("Missing values after cleaning:")
print(df.isna().sum().sort_values(ascending=False).head(10))

Loaded 571 players and 59 columns
Duplicate rows: 0
Missing values after cleaning:
Throw outs            502
Sweeper clearances    502
Catches               502
High Claims           502
Punches               502
Penalties saved       502
Saves                 502
Goal Kicks            502
Clean sheets          309
Goals conceded        309
dtype: int64


In [4]:
# Derived indicators use appearances as the exposure denominator.
appearances = df["Appearances"].replace(0, np.nan)
shots = df["Shots"].replace(0, np.nan)

# New analytical columns.
df["Goals per appearance (clean)"] = (df["Goals"] / appearances).round(3)
df["Win rate"] = (df["Wins"] / appearances).round(3)
df["Loss rate"] = (df["Losses"] / appearances).round(3)
df["Shot conversion rate"] = (df["Goals"] / shots).round(3)
df["Shots on target rate"] = (df["Shots on target"] / shots).round(3)
df["Assists per appearance"] = (df["Assists"] / appearances).round(3)
df["Goal contributions"] = df["Goals"] + df["Assists"]
df["Defensive actions"] = (
    df["Tackles"] + df["Interceptions"] + df["Clearances"] + df["Recoveries"]
)
df["Defensive actions per appearance"] = (
    df["Defensive actions"] / appearances
).round(3)
df["Discipline points"] = df["Yellow cards"] + (2 * df["Red cards"])
df["Discipline points per appearance"] = (
    df["Discipline points"] / appearances
).round(3)

# A conservative quality flag makes missing or invalid source records visible.
df["Data quality flag"] = np.select(
    [
        df[["Name", "Club", "Position"]].isna().any(axis=1),
        df["Age"].notna() & ~df["Age"].between(15, 50),
        df["Appearances"].lt(0) | df["Goals"].lt(0),
    ],
    ["missing identity", "invalid age", "invalid count"],
    default="ok",
)

# Ranking is only meaningful for players with at least five appearances.
eligible = df["Appearances"].ge(5)
attack_rate = (df["Goals"] + df["Assists"]) / appearances
win_rate = df["Win rate"]
df["Impact score"] = np.nan
df.loc[eligible, "Impact score"] = (
    0.6 * attack_rate[eligible].rank(pct=True)
    + 0.4 * win_rate[eligible].rank(pct=True)
).round(3)

print("New columns:")
print(df.columns[-15:].tolist())
print("Quality flags:")
print(df["Data quality flag"].value_counts(dropna=False).to_dict())
print("Top impact scores:")
print(df.loc[eligible].nlargest(10, "Impact score")[["Name", "Club", "Position", "Impact score"]].to_string(index=False))

New columns:
['Fouls', 'Offsides', 'Goals per appearance (clean)', 'Win rate', 'Loss rate', 'Shot conversion rate', 'Shots on target rate', 'Assists per appearance', 'Goal contributions', 'Defensive actions', 'Defensive actions per appearance', 'Discipline points', 'Discipline points per appearance', 'Data quality flag', 'Impact score']
Quality flags:
{'ok': 571}
Top impact scores:
           Name              Club   Position  Impact score
  Mohamed Salah         Liverpool    Forward         0.983
Kevin De Bruyne   Manchester-City Midfielder         0.975
  Gabriel Jesus   Manchester-City    Forward         0.975
  Sergio Agüero   Manchester-City    Forward         0.974
Bruno Fernandes Manchester-United Midfielder         0.960
     Sadio Mané         Liverpool    Forward         0.958
Roberto Firmino         Liverpool    Forward         0.957
Raheem Sterling   Manchester-City    Forward         0.947
  Son Heung-Min Tottenham-Hotspur    Forward         0.941
     Harry Kane Tottenham

In [5]:
# Reusable cleaned table and compact insight summaries.
clean_df = df.copy()

club_summary = (
    clean_df.groupby("Club", as_index=False)
    .agg(
        Players=("Name", "count"),
        Total_goals=("Goals", "sum"),
        Total_assists=("Assists", "sum"),
        Average_win_rate=("Win rate", "mean"),
    )
    .sort_values(["Total_goals", "Total_assists"], ascending=False)
)

position_summary = (
    clean_df.groupby("Position", as_index=False)
    .agg(
        Players=("Name", "count"),
        Average_age=("Age", "mean"),
        Average_impact=("Impact score", "mean"),
        Total_goal_contributions=("Goal contributions", "sum"),
    )
    .sort_values("Average_impact", ascending=False)
)

# Basic checks protect the cleaned table from silent calculation errors.
assert len(clean_df) == len(raw_df)
assert clean_df["Data quality flag"].eq("ok").all()
assert clean_df["Win rate"].dropna().between(0, 1).all()
assert clean_df["Shot conversion rate"].dropna().between(0, 1).all()

print("Top clubs by total goals:")
print(club_summary.head(10).to_string(index=False))
print("\nSummary by position:")
print(position_summary.to_string(index=False))
print(f"\nClean table shape: {clean_df.shape}")

Top clubs by total goals:
             Club  Players  Total_goals  Total_assists  Average_win_rate
  Manchester-City       27          496            313          0.682750
        Liverpool       34          447            367          0.630179
Tottenham-Hotspur       32          387            221          0.462080
   Crystal-Palace       31          359            194          0.354120
          Everton       31          298            232          0.398724
Manchester-United       31          270            209          0.468133
   Leicester-City       31          269            183          0.419414
 Newcastle-United       30          234            164          0.310704
          Arsenal       30          231            202          0.483148
          Chelsea       27          219            159          0.522250

Summary by position:
  Position  Players  Average_age  Average_impact  Total_goal_contributions
   Forward      110    25.372727        0.640161                      3300